# 🩺 Medical On-Device LLM: Fine-Tuning, Inference Testing & GGUF Export
This notebook trains your model on the clinical dataset, runs live test prompts to display generated outputs, and exports the model to **GGUF** for Flutter.

### 📍 Step 1: Set Working Directory
Ensures Colab is working in the `medtrain` directory.

In [1]:
import os
if os.path.exists('/content/medtrain'):
    %cd /content/medtrain
elif os.path.exists('medtrain'):
    %cd medtrain

print('Current working directory:', os.getcwd())

Current working directory: /content


### ⚡ Step 2: Check GPU Hardware Acceleration

In [2]:
!nvidia-smi

Sun Sep 27 12:25:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### 📦 Step 3: Install Required Dependencies

In [3]:
!pip install -q --upgrade pip
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 74.6 MB/s eta 0:00:00
ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'


### 🧪 Step 4: Verify Database & Schema Sanity

In [4]:
!python test_db_and_ai.py

python3: can't open file '/content/test_db_and_ai.py': [Errno 2] No such file or directory


### 🚀 Step 5: Train the Model on Category 1 Dataset
Trains on `categories/cat_01_patient_creation/data.jsonl` (or your chosen dataset), merges LoRA adapters, and converts to `model.gguf`.

In [5]:
!python train.py \
  --dataset_path categories/cat_01_patient_creation/data.jsonl \
  --epochs 3 \
  --batch_size 4 \
  --lr 2e-4 \
  --gguf_output ./model.gguf

python3: can't open file '/content/train.py': [Errno 2] No such file or directory


### 🔍 Step 6: Test Model Inference & Display Generated Outputs
Feeds unseen patient registration utterances to the model and displays the extracted DSL commands and SAY responses.

In [ ]:
!python test_inference.py --model_path ./medical_merged_model

### 💬 Step 7: Interactive Live Prompt Playground (Try Your Own Text!)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model for quick interactive testing
model_path = './medical_merged_model'
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16, device_map='auto')

def ask_model(user_query):
    prompt = (
        f"<|im_start|>system\nYou are a clinical AI assistant that parses consultations into database actions (ADD_PATIENT, LOG_VISIT, etc.) followed by a SAY: response.<|im_end|>\n"
        f"<|im_start|>user\n{user_query}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=120, temperature=0.2, pad_token_id=tokenizer.eos_token_id)
    print('USER:', user_query)
    print('AI OUTPUT:')
    print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

# Test custom query
ask_model('Please register new patient Daniel Craig, born 1968-03-02, male, blood group O negative.')

### 💾 Step 8: Download `model.gguf` to Your Local Machine

In [ ]:
import os
from google.colab import files

if os.path.exists('model.gguf'):
    mb = os.path.getsize('model.gguf') / (1024 * 1024)
    print(f'Downloading model.gguf ({mb:.2f} MB)...')
    files.download('model.gguf')
else:
    print('model.gguf not found.')